In [1]:
!pip install -q transformers datasets nltk

In [2]:
import os
import torch
import pandas as pd
import numpy as np
from transformers import (
    AutoTokenizer,
    BartForConditionalGeneration,  # BART's generation model, different from BERT's classification
    AutoModelForSequenceClassification,  # BERT's classification model
    BartTokenizer,
    Seq2SeqTrainer,                # specialized trainer for sequence to sequence tasks like summarization
    Seq2SeqTrainingArguments,      # training arguments for seq2seq models
    DataCollatorForSeq2Seq         # handles padding for seq2seq batches
)
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
import nltk
from nltk.tokenize import sent_tokenize
nltk.download('averaged_perceptron_tagger_eng', quiet=True)  # one-time download
nltk.download('punkt_tab', quiet=True)

from google.colab import drive

# Mount drive so we can access our CSVs and save our model
drive.mount('/content/drive')
base_path = '/content/drive/MyDrive/Colab Notebooks/ToS_summarizer'

# Same GPU check as before
device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(f"Using device: {device}")

Mounted at /content/drive
Using device: cuda


In [3]:
# Load our preprocessed splits from Drive
df_train = pd.read_csv(f'{base_path}/df_train.csv')
df_val = pd.read_csv(f'{base_path}/df_val.csv')
df_test = pd.read_csv(f'{base_path}/df_test.csv')

# We'll use this to convert numeric labels back to human readable strings
# These will be prepended to the input so BART knows what kind of clause it's summarizing
id2label = {
    0: 'clearly_fair',
    1: 'potentially_unfair',
    2: 'clearly_unfair'
}

print(f"Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")
print(df_train.head())


Train: 7470 | Val: 971 | Test: 960
                                            sentence    unfairness_level  \
0  these terms and any rights and licenses grante...        clearly_fair   
1  the user is responsible for all damages liabil...      clearly_unfair   
2  no refunds for downtime  the company is not li...  potentially_unfair   
3  ea recommends that parents and guardians famil...        clearly_fair   
4  the company can limit or restrict your ability...  potentially_unfair   

   label  
0      0  
1      2  
2      1  
3      0  
4      1  


In [6]:
# to check if the snippet has a verb in it and filter out those that don't for better results
def has_verb(sentence):
    """Returns True if the sentence contains at least one verb —
    a decent proxy for 'this is a real clause, not a bare heading'."""
    tokens = nltk.word_tokenize(sentence)
    tags = nltk.pos_tag(tokens)
    return any(tag.startswith('VB') for word, tag in tags)

# Apply to a copy used specifically for BART's distillation targets —
# doesn't touch df_train used for BERT classification
df_train_for_bart = df_train[df_train['sentence'].apply(has_verb)].copy()

print(f"Original rows: {len(df_train)}")
print(f"After header/fragment filter: {len(df_train_for_bart)}")
print(f"Dropped: {len(df_train) - len(df_train_for_bart)}")


Original rows: 7470
After header/fragment filter: 7104
Dropped: 366


In [7]:
df_val_for_bart = df_val[df_val['sentence'].apply(has_verb)].copy()

print(f"Original val rows: {len(df_val)}")
print(f"After header/fragment filter: {len(df_val_for_bart)}")
print(f"Dropped: {len(df_val) - len(df_val_for_bart)}")

Original val rows: 971
After header/fragment filter: 916
Dropped: 55


In [8]:
# Load the supplemental summarization dataset
summary_dataset = load_dataset("EE21/ToS-Summaries")

REPLACEMENT = "the applicable jurisdiction"

def clean_placeholder(example):
    example["summary"] = example["summary"].replace("location X", REPLACEMENT)
    return example

# Convert to DataFrame and inspect
df_summaries = pd.DataFrame(summary_dataset['train'])

df_summaries["summary"] = df_summaries["summary"].str.replace("location X", REPLACEMENT, regex=False)

remaining = df_summaries["summary"].str.contains("location X").sum()
print(f"Remaining 'location X' occurrences: {remaining}")

# See what we're working with
print(df_summaries.shape)
print(df_summaries.columns.tolist())
print(df_summaries.head(3))

README.md:   0%|          | 0.00/128 [00:00<?, ?B/s]

dataset.json:   0%|          | 0.00/4.14M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/901 [00:00<?, ? examples/s]

Remaining 'location X' occurrences: 0
(901, 2)
['plain_text', 'summary']
                                          plain_text  \
0  We can change these Terms at any time. We keep...   
1  How To File a DMCA Notice To submit a notice o...   
2  You can see our previous Privacy Policy    her...   

                                             summary  
0  Users should revisit the terms periodically, a...  
1  This service will aid you when other users inf...  
2  There is a date of the last update of the agre...  


In [3]:
# Load our saved BERT model from Drive to label the supplemental dataset
bert_tokenizer = AutoTokenizer.from_pretrained(f'{base_path}/bert_model')
bert_model = AutoModelForSequenceClassification.from_pretrained(f'{base_path}/bert_model')
bert_model = bert_model.to(device)
bert_model.eval()

print("BERT model loaded!")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BERT model loaded!


In [10]:
def run_inference(df, model, tokenizer, device):
    predictions = []

    for sentence in df['sentence']:
        # Tokenize the sentence
        inputs = tokenizer(
            str(sentence),
            max_length=256,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        ).to(device)

        # Get prediction
        with torch.no_grad():
            outputs = model(**inputs)
            pred = torch.argmax(outputs.logits, dim=1).item()
            predictions.append(id2label[pred])

    df['bert_label'] = predictions
    return df

# Rename plain_text to sentence so it matches our existing DataFrames
df_summaries = df_summaries.rename(columns={'plain_text': 'sentence'})

# Run BERT inference on the summaries dataset to get unfairness labels
print("Running BERT inference on summaries dataset...")
df_summaries = run_inference(df_summaries, bert_model, bert_tokenizer, device)

# Sanity check the label distribution
print(df_summaries['bert_label'].value_counts())
print(df_summaries.head(3))

Running BERT inference on summaries dataset...
bert_label
clearly_fair          589
potentially_unfair    258
clearly_unfair         54
Name: count, dtype: int64
                                            sentence  \
0  We can change these Terms at any time. We keep...   
1  How To File a DMCA Notice To submit a notice o...   
2  You can see our previous Privacy Policy    her...   

                                             summary          bert_label  
0  Users should revisit the terms periodically, a...  potentially_unfair  
1  This service will aid you when other users inf...        clearly_fair  
2  There is a date of the last update of the agre...        clearly_fair  


In [11]:
# Load the BART tokenizer
# facebook/bart-large-cnn is pretrained on news summarization
# making it a great starting point for our legal clause summarization
tokenizer = BartTokenizer.from_pretrained('facebook/bart-large-cnn')

# Reuse the already-cleaned df_summaries from Cell 5 — do NOT reload EE21 here,
# that would wipe out the "location X" placeholder fix
df_summaries = df_summaries.rename(columns={'plain_text': 'sentence'})

# Run BERT inference on supplemental dataset to get unfairness labels
print("Running BERT inference on summaries dataset...")
df_summaries = run_inference(df_summaries, bert_model, bert_tokenizer, device)

print(df_summaries['bert_label'].value_counts())
print(df_summaries.head(3))

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Running BERT inference on summaries dataset...
bert_label
clearly_fair          589
potentially_unfair    258
clearly_unfair         54
Name: count, dtype: int64
                                            sentence  \
0  We can change these Terms at any time. We keep...   
1  How To File a DMCA Notice To submit a notice o...   
2  You can see our previous Privacy Policy    her...   

                                             summary          bert_label  
0  Users should revisit the terms periodically, a...  potentially_unfair  
1  This service will aid you when other users inf...        clearly_fair  
2  There is a date of the last update of the agre...        clearly_fair  


In [12]:
# Load the real distilled summaries generated via the batch job earlier —
# these replace the old "label: sentence" placeholder entirely
df_train_for_bart = pd.read_csv(f'{base_path}/df_train_bart_distilled.csv')
df_val_for_bart = pd.read_csv(f'{base_path}/df_val_bart_distilled.csv')

# Align EE21 supplemental data to the same column structure
df_summaries['unfairness_level'] = df_summaries['bert_label']
df_summaries['label'] = df_summaries['bert_label'].map({
    'clearly_fair': 0,
    'potentially_unfair': 1,
    'clearly_unfair': 2
})

# Combine: real distilled TOSDatasetV3 summaries + real EE21 summaries
df_train_final = pd.concat(
    [df_train_for_bart, df_summaries[['sentence', 'unfairness_level', 'label', 'summary']]],
    ignore_index=True
)
df_val_final = df_val_for_bart.copy()

print(f"Train rows: {len(df_train_final)}")
print(f"Val rows: {len(df_val_final)}")
print(df_train_final[['sentence', 'unfairness_level', 'summary']].head(3))

Train rows: 8005
Val rows: 916
                                            sentence    unfairness_level  \
0  these terms and any rights and licenses grante...        clearly_fair   
1  the user is responsible for all damages liabil...      clearly_unfair   
2  no refunds for downtime  the company is not li...  potentially_unfair   

                                             summary  
0  You cannot transfer or assign these terms or a...  
1  You have to pay for all damages, liabilities, ...  
2  We won't give you your money back if our servi...  


In [13]:
class TOSSummarizationDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_input_length=256, max_target_length=128):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_input_length = max_input_length
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]

        # Label now actually gets prepended to the input (this was a
        # comment-only claim before — never really happened)
        input_text = f"{row['unfairness_level']}: {row['sentence']}"

        # Real plain-English target, not a label+sentence echo
        target_text = str(row['summary'])

        input_encoding = self.tokenizer(
            input_text, max_length=self.max_input_length,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        target_encoding = self.tokenizer(
            target_text, max_length=self.max_target_length,
            padding='max_length', truncation=True, return_tensors='pt'
        )

        labels = target_encoding['input_ids'].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            'input_ids': input_encoding['input_ids'].squeeze(),
            'attention_mask': input_encoding['attention_mask'].squeeze(),
            'labels': labels
        }

In [14]:
# Instantiate our dataset class for each split
train_dataset = TOSSummarizationDataset(df_train_final, tokenizer)
val_dataset = TOSSummarizationDataset(df_val_final, tokenizer)
# test_dataset = TOSSummarizationDataset(df_test, tokenizer)


# DataLoaders handle batching during training
# batch size of 4 because BART is much larger than BERT and needs more memory per sample
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    pin_memory=True if torch.cuda.is_available() else False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    pin_memory=True if torch.cuda.is_available() else False
)

# test_loader = DataLoader(
#     test_dataset,
#     batch_size=4,
#     shuffle=False,
#     pin_memory=True if torch.cuda.is_available() else False
# )

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
# print(f"Test batches: {len(test_loader)}")

Train batches: 2002
Val batches: 229


In [15]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
# Load the BART model pretrained on CNN/DailyMail news summarization
# This gives us a great starting point since it already knows how to summarize
model = BartForConditionalGeneration.from_pretrained('facebook/bart-large-cnn')

# Move model to GPU if available
model = model.to(device)

# Set up optimizer — same AdamW as BERT but lower learning rate
# BART is bigger and more sensitive so we want smaller updates
optimizer = AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)

# Calculate total training steps for the scheduler
total_steps = len(train_loader) * 3  # 3 epochs

# Same warmup scheduler as BERT
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

print(f"BART model loaded on: {device}")
print(f"Total training steps: {total_steps}")

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

BART model loaded on: cuda
Total training steps: 6006


In [16]:
def train_bart_epoch(model, dataloader, optimizer, scheduler, device):
    # Set model to training mode
    model.train()
    total_loss = 0

    for batch in dataloader:
        # Move batch to GPU
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Zero gradients
        optimizer.zero_grad()

        # Forward pass — BART computes loss internally when labels are provided
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        # Backward pass
        loss.backward()

        # Clip gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # Update weights and scheduler
        optimizer.step()
        scheduler.step()

    return total_loss / len(dataloader)


def eval_bart_epoch(model, dataloader, device):
    # Set model to evaluation mode
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            total_loss += outputs.loss.item()

    return total_loss / len(dataloader)

In [17]:
EPOCHS = 3
best_val_loss = float('inf')

for epoch in range(EPOCHS):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print(f"{'='*50}")

    # Run training pass
    train_loss = train_bart_epoch(
        model, train_loader, optimizer, scheduler, device
    )

    # Run validation pass
    val_loss = eval_bart_epoch(
        model, val_loader, device
    )

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")

    # Save model checkpoint if validation loss improved
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        new_model_path = f'{base_path}/bart_model_v2'
        model.save_pretrained(new_model_path)
        tokenizer.save_pretrained(new_model_path)
        print(f"✅ Model improved and saved to Drive!")
    else:
        print(f"⚠️ No improvement this epoch")

print("\nTraining complete!")


Epoch 1/3
Train Loss: 1.0694
Val Loss:   0.8006


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model improved and saved to Drive!

Epoch 2/3
Train Loss: 0.5908
Val Loss:   0.7980


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model improved and saved to Drive!

Epoch 3/3
Train Loss: 0.4388
Val Loss:   0.7996
⚠️ No improvement this epoch

Training complete!


### STOP HERE FOR NOW, JUST NEED TO TRAIN

In [4]:
# Load the saved BART model and tokenizer from Drive
# This ensures we're using the best checkpoint rather than the current training state

bart_model = BartForConditionalGeneration.from_pretrained(f'{base_path}/bart_model_v2')
bart_tokenizer = BartTokenizer.from_pretrained(f'{base_path}/bart_model_v2')

bart_model = bart_model.to(device)
bart_model.eval()

print("✅ BART model loaded from Drive!")

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

✅ BART model loaded from Drive!


In [5]:
def clean_df(df):
    df = df[df['sentence'].str.len() >= 20].copy()
    df['sentence'] = df['sentence'].str.strip()
    df = df[df['sentence'].str.len() > 0]
    return df

In [6]:
def load_tos_document(file_path):
    # Handles .txt files for now
    # Can be extended to handle .pdf or .docx later
    if file_path.endswith('.txt'):
        with open(file_path, 'r', encoding='utf-8') as f:
            text = f.read()
    else:
        raise ValueError("Currently only .txt files are supported!")

    # Convert to DataFrame so we can run clean_df on it
    # We split on newlines to get individual lines first
    df = pd.DataFrame({'sentence': [s.strip() for s in text.split('\n') if len(s.strip()) > 0]})

    # Run our preprocessing cleaning function
    df = clean_df(df)

    # Return as a single clean string to pass into process_tos_document
    return ' '.join(df['sentence'].tolist())

# To use with a real document instead of the sample string,
# upload your ToS file to Colab and replace the sample_tos variable like this:
# document = load_tos_document('/content/your_tos_file.txt')
# results = process_tos_document(document, bert_model, model, bert_tokenizer, tokenizer, device)

In [9]:
id2label = {
    0: 'clearly_fair',
    1: 'potentially_unfair',
    2: 'clearly_unfair'
}

In [12]:
def process_tos_document(document, bert_model, bart_model, bert_tokenizer, bart_tokenizer, device):
    # Step 1 — Split document into individual clauses by sentence
    # We use a simple split on punctuation for now
    clauses = [s.strip() for s in document.replace('\n', ' ').split('.') if len(s.strip()) >= 20]

    print(f"Found {len(clauses)} clauses in document")

    # Step 2 — Run BERT classification on each clause
    bert_model.eval()
    flagged_clauses = []

    for clause in clauses:
        inputs = bert_tokenizer(
            clause,
            max_length=256,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        ).to(device)

        with torch.no_grad():
            outputs = bert_model(**inputs)
            pred = torch.argmax(outputs.logits, dim=1).item()
            label = id2label[pred]

        # Step 3 — Filter out clearly fair clauses, keep the rest
        if label != 'clearly_fair':
            flagged_clauses.append((clause, label))

    print(f"Flagged {len(flagged_clauses)} potentially risky clauses")

    # Step 4 — Run BART summarization on flagged clauses
    bart_model.eval()
    results = []

    for clause, label in flagged_clauses:
        # Match the input format BART was trained on: "{label}: {clause}"
        input_text = f"{label}: {clause}"

        inputs = bart_tokenizer(
            input_text,
            max_length=256,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        ).to(device)

        with torch.no_grad():
            # Generate summary
            summary_ids = bart_model.generate(
                inputs['input_ids'],
                attention_mask=inputs['attention_mask'],
                max_length=128,
                min_length=20,
                num_beams=4,        # beam search for better quality output
                length_penalty=2.0, # encourages longer summaries
                early_stopping=True
            )

        summary = bart_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

        # Strip any label prefixes that leaked from placeholder training data
        for prefix in ['clearly_fair: ', 'clearly_unfair: ', 'potentially_unfair: ']:
            summary = summary.replace(prefix, '')

        results.append((label, summary))

    # Step 5 — Format and print plain English output
    print("\n" + "="*50)
    print("TERMS OF SERVICE ANALYSIS")
    print("="*50)

    for label, summary in results:
        if label == 'clearly_unfair':
            emoji = '🚨'
        else:
            emoji = '⚠️'

        print(f"\n{emoji} {label.upper().replace('_', ' ')}")
        print(f"• {summary}")

    print("\n" + "="*50)
    return results


# Test it with a sample clause once BART finishes training
sample_tos = """
The company reserves the right to terminate your account at any time without notice.
Users are responsible for all charges incurred under their account.
We may share your personal data with third party partners for marketing purposes.
You agree to receive promotional emails from us and our partners.
"""

results = process_tos_document(
    sample_tos,
    bert_model,
    bart_model,  # this is our BART model
    bert_tokenizer,
    bart_tokenizer,  # this is our BART tokenizer
    device
)

Found 4 clauses in document
Flagged 2 potentially risky clauses

TERMS OF SERVICE ANALYSIS

⚠️ POTENTIALLY UNFAIR
• The company can close your account whenever it wants without telling you first, and doesn't have to tell you first.

⚠️ POTENTIALLY UNFAIR
• You are responsible for paying all charges that happen on your account, no matter what.



In [13]:
# Load and process a real ToS document
document = load_tos_document(f'{base_path}/fb_tos_testing.txt')

# Run the full pipeline!
results = process_tos_document(
    document,
    bert_model,
    bart_model,
    bert_tokenizer,
    bart_tokenizer,
    device
)

Found 190 clauses in document
Flagged 39 potentially risky clauses

TERMS OF SERVICE ANALYSIS

🚨 CLEARLY UNFAIR
• These Terms are an agreement between you and Meta Platforms, Inc., and you agree to follow them.

🚨 CLEARLY UNFAIR
• These Terms (formerly called the Statement of Rights and Responsibilities) are the complete and final agreement between you and Meta Platforms, Inc.

🚨 CLEARLY UNFAIR
• What products are covered by these Terms? What type of products do these Terms apply to? 1

🚨 CLEARLY UNFAIR
• Section 2 explains this in more detail, so we can explain it more clearly.

⚠️ POTENTIALLY UNFAIR
• If we find out about content or behavior like this, we can take appropriate action based on our assessment, which might include notifying you, offering help, removing content, removing or restricting access to certain features, disabling an account, or contacting law enforcement.

⚠️ POTENTIALLY UNFAIR
• The Oversight Board can review our decisions, but only if its terms and bylaws allo

In [14]:
import pandas as pd

df_train_check = pd.read_csv(f'{base_path}/df_train_bart_distilled.csv')
df_val_check = pd.read_csv(f'{base_path}/df_val_bart_distilled.csv')

# Telltale phrases that indicate a refusal/meta-commentary got saved as a "summary"
# instead of an actual plain-English rewrite
refusal_markers = [
    "I can see this",
    "I don't see",
    "I'd be happy to help",
    "Could you please provide",
    "no substantive",
    "section heading rather than",
    "does not contain",
    "appears to be a",
    "please provide the",
]

def looks_like_refusal(text):
    text = str(text).lower()
    return any(marker.lower() in text for marker in refusal_markers)

train_bad = df_train_check[df_train_check['summary'].apply(looks_like_refusal)]
val_bad = df_val_check[df_val_check['summary'].apply(looks_like_refusal)]

print(f"Train rows flagged: {len(train_bad)}")
print(f"Val rows flagged: {len(val_bad)}")
print("\n--- Flagged rows ---")
for _, row in pd.concat([train_bad, val_bad]).iterrows():
    print(f"ORIGINAL: {row['sentence']}")
    print(f"SUMMARY:  {row['summary']}")
    print("-" * 80)

Train rows flagged: 46
Val rows flagged: 5

--- Flagged rows ---
ORIGINAL: registered offices park view   the vale london wrw
SUMMARY:  I can see this appears to be an address fragment rather than a complete clause with legal content. Based on what's provided, here's the plain-English rewrite:

Registered offices are located at Park View, The Vale, London, WRW.
--------------------------------------------------------------------------------
ORIGINAL: any clause that requires the customer to give up their right to privacy or confidentiality
SUMMARY:  We don't have a clause to rewrite here—you've provided a description of what type of clause to look for, rather than an actual clause from a Terms of Service. Please provide the specific text you'd like rewritten.
--------------------------------------------------------------------------------
ORIGINAL: a carrier  radosaw rzepnikowski and janusz sadowski acting as trans kinetik sp
SUMMARY:  I can't rewrite this clause because it appears to 